In [ ]:
session.sql("DROP TABLE IF EXISTS tmp_custodies_stream_snapshot").collect()
print("Temp table dropped")

In [ ]:
session.sql("""
CREATE TEMP TABLE tmp_custodies_stream_snapshot AS
SELECT *
FROM STREAM_T_CUSTODIES
WHERE METADATA$ACTION = 'INSERT'
""").collect()

cus_raw = session.table("tmp_custodies_stream_snapshot")
cus_raw = cus_raw.drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID")
print(f"snapshot custodies records found")

In [ ]:
ind_columns = ["TWENTY_NINE_C_IND"]

cus_clean = cus_raw
for c in ind_columns:
    cus_clean = cus_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("N")
        ).otherwise(trim(col(c)))
    )

code_columns = ["LEGAL_STATUS_CODE", "END_REASON_CODE", "CUSTODY_TYPE"]

for c in code_columns:
    cus_clean = cus_clean.with_column(
        c,
        upper(trim(col(c)))
    )

user_columns = ["ADD_USER", "MOD_USER"]

for c in user_columns:
    cus_clean = cus_clean.with_column(
        c,
        when(
            (trim(col(c)).is_null()) | (trim(col(c)) == lit("")),
            lit("SYSTEM")
        ).otherwise(trim(col(c)))
    )

trim_only_columns = ["SOURCE_FILE_NAME"]

for c in trim_only_columns:
    cus_clean = cus_clean.with_column(
        c,
        trim(col(c))
    )

cus_clean = cus_clean.with_column_renamed("LOAD_TIMESTAMP", "RAW_LOAD_TIMESTAMP")
print("cus clean")

In [ ]:
valid_cus = cus_clean.filter(
    col("CUS_ID").is_not_null()
)

invalid_cus = cus_clean.filter(
    col("CUS_ID").is_null()
).with_column(
    "ERROR_REASON",
    lit("CUS_ID_NULL")
)
print("seperated valid and invalid records")

In [ ]:
checksum_columns = [
    ("CUS_ID", "number"),
    ("LEGAL_STATUS_CODE", "string"),
    ("TWENTY_NINE_C_IND", "string"),
    ("CUSTODY_TYPE", "string"),
    ("END_REASON_CODE", "string"),
    ("ADD_USER", "string"),
    ("MOD_USER", "string"),
    ("PERSON_PERSON_ID", "number"),
    ("ORGN_ORGN_ID", "number"),
    ("VPA_VPA_ID", "number"),
    ("SUR_SUR_ID", "number"),
    ("CAP_CAP_ID", "number"),
    ("POI_POI_ID", "number"),
    ("PERSON_PERSON_CHILD_ID", "number"),
    ("TICKLER_ID", "number"),
    ("MOD_CAP_CAP_ID", "number"),
    ("START_DATE", "timestamp"),
    ("END_DATE", "timestamp"),
    ("CHILD_ADJUDICATED_DATE", "timestamp"),
    ("TWENTY_NINE_C_DATE", "timestamp"),
    ("REASONABLE_EFFORT_DATE", "timestamp"),
]

checksum_exprs = []
for col_name, col_type in checksum_columns:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

cus_final = valid_cus.with_column(
    "CHECKSUM",
    sha2(concat_ws(lit("|"), *checksum_exprs), 256)
)

In [ ]:
cus_final = cus_final.with_column(
    "STAGING_LOADED_TIMESTAMP",
    current_timestamp()
)

cus_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_CUSTODIES}",
    mode="truncate"
)

print(f"CUSTODIES loaded into {STG}.{STG_CUSTODIES}")

In [ ]:
invalid_cus.create_or_replace_temp_view("tmp_invalid_cus")

invalid_count = invalid_cus.count()

if invalid_count > 0:
    session.sql(f"""
    INSERT INTO {DB}.{STG}.{INVALID_RECORDS}
    (
        TABLE_NAME,
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        RAW_RECORD
    )
    SELECT
        'T_CUSTODIES',
        ERROR_REASON,
        SOURCE_FILE_NAME,
        RAW_LOAD_TIMESTAMP,
        OBJECT_CONSTRUCT(*)
    FROM tmp_invalid_cus
    """).collect()
    print(f"Invalid records saved into {STG}.{INVALID_RECORDS}")
else:
    print("No invalid records")

In [ ]:
rows_processed = cus_final.count()
rows_failed = invalid_count

status = 'SUCCESS' if rows_failed == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    session.sql("SELECT UUID_STRING()").collect()[0][0],
    'STG_CUSTODIES_LOAD',
    'STAGING',
    datetime.now(),
    datetime.now(),
    rows_processed,
    rows_failed,
    status,
    None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status,
    'STG_CUSTODIES_LOAD',
    'STAGING',
    f'CUSTODIES staging completed. Rows processed: {rows_processed}, Failed rows: {rows_failed}'
)
print("Audit log inserted and email sent")